In [51]:
import pandas as pd
import numpy as np

# ─────────────────────────────────────────────────────────────────
# 1. LEER EL CSV
# Pon el archivo en la misma carpeta que este notebook
# ─────────────────────────────────────────────────────────────────
df_raw = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

print(f'Dataset original: {df_raw.shape[0]} clientes × {df_raw.shape[1]} columnas')
print()
cols_vista = ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn']
print('Primeras 3 filas (columnas clave):')
print(df_raw[cols_vista].head(3).to_string())

# ─────────────────────────────────────────────────────────────────
# 2. CORREGIR TotalCharges
# En el CSV original esta columna es string (tiene espacios vacíos)
# pd.to_numeric con errors='coerce' convierte los invalidos en NaN
# ─────────────────────────────────────────────────────────────────
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')
print(f'\nFilas con TotalCharges nulo: {df_raw["TotalCharges"].isnull().sum()} (las eliminaremos)')

# ─────────────────────────────────────────────────────────────────
# 3. FEATURE ENGINEERING — num_services
# Contamos cuantos servicios adicionales tiene cada cliente
# Analogia a 'num_categorias' del dataset de clase
# ─────────────────────────────────────────────────────────────────
servicios = ['PhoneService', 'OnlineSecurity', 'OnlineBackup',
             'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies']

# Para cada columna de servicio: Yes→1, No→0
# axis=1 suma por fila (→) → un total por cliente
df_raw['num_services'] = df_raw[servicios].apply(
    lambda col: (col == 'Yes').astype(int)
).sum(axis=1)

# ─────────────────────────────────────────────────────────────────
# 4. SELECCIÓN DE FEATURES Y LIMPIEZA
# ─────────────────────────────────────────────────────────────────
FEATURES = ['tenure', 'MonthlyCharges', 'TotalCharges', 'num_services', 'SeniorCitizen']
TARGET   = 'Churn'

df = df_raw[FEATURES + [TARGET]].dropna().copy()
df['Churn_num'] = (df[TARGET] == 'Yes').astype(float)  # 'Yes'→1.0, 'No'→0.0

# ─────────────────────────────────────────────────────────────────
# 5. MATRICES NUMPY — lo que el modelo realmente recibe
# ─────────────────────────────────────────────────────────────────
X_raw = df[FEATURES].values.astype(float)
y     = df['Churn_num'].values

print(f'\nDataset limpio: {df.shape[0]} clientes, {len(FEATURES)} features')
print(f'X_raw.shape = {X_raw.shape}   ← (clientes, features)')
print(f'y.shape     = {y.shape}       ← un numero por cliente')
print()
print(f'Se fueron   (Churn=1): {int(y.sum()):4d}  ({y.mean()*100:.1f}%)')
print(f'Se quedaron (Churn=0): {int((y==0).sum()):4d}  ({(1-y.mean())*100:.1f}%)')

Dataset original: 7043 clientes × 21 columnas

Primeras 3 filas (columnas clave):
   customerID  tenure  MonthlyCharges TotalCharges  SeniorCitizen Churn
0  7590-VHVEG       1           29.85        29.85              0    No
1  5575-GNVDE      34           56.95       1889.5              0    No
2  3668-QPYBK       2           53.85       108.15              0   Yes

Filas con TotalCharges nulo: 11 (las eliminaremos)

Dataset limpio: 7032 clientes, 5 features
X_raw.shape = (7032, 5)   ← (clientes, features)
y.shape     = (7032,)       ← un numero por cliente

Se fueron   (Churn=1): 1869  (26.6%)
Se quedaron (Churn=0): 5163  (73.4%)


In [52]:
np.set_printoptions(suppress=True, precision=2)

In [53]:
df.info()

<class 'pandas.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   tenure          7032 non-null   int64  
 1   MonthlyCharges  7032 non-null   float64
 2   TotalCharges    7032 non-null   float64
 3   num_services    7032 non-null   int64  
 4   SeniorCitizen   7032 non-null   int64  
 5   Churn           7032 non-null   str    
 6   Churn_num       7032 non-null   float64
dtypes: float64(3), int64(3), str(1)
memory usage: 439.5 KB


¿Qué es un Escalar? : 0 dimensiones, IA: Resultado de una predicción, El precio de un producto.
¿Qué es un vector? : 1D, Una fila del dataset, una colección, 
¿Qué es una Matriz? : 2D, Multiples clientes
¿Qué es un Tensor?: 3D+, Una imagen, un cubo de datos, Matrices apiladas. Un cubo de datos, la profundidad del tiempo.

In [54]:
#Escalares
cargo_mensual = np.array(29.85)
print(cargo_mensual.shape)
print(cargo_mensual.ndim)

()
0


In [55]:
#Vector es una lista de numeros en linea recta (fila), tiene longitud pero en una sola dirección
#                    tenure  MonthlyCharges  total  services  senior
cli_01 = np.array([1,          29.85,       29.85,   1,         0])
cli_01[0]
cli_01[1]
cli_01[1:3] # Inicio:fin, incluye el inicio, excluye el fin. Venga deme desde la posicion pero antes del 3
cli_01[-1] # El ultimo elemento
print(cli_01.shape)
print(cli_01.ndim)


(5,)
1


In [56]:
#Matrices
clientes = np.array([
    #tenure   MonthlyCharges  total   services   senior
    [1,          29.85,       29.85,   1,        0],
    [34,         56.85,       1500.85, 3,        0],
    [2,          45.85,       100.85,  3,        0],
    [45,         60.85,       2000.85, 5,        1],
    [3,          70.85,       150.85,  4,        0],
    
])

In [57]:
print(clientes.shape)
print(clientes.ndim)

(5, 5)
2


In [58]:
clientes[0, :] #traigame todo
clientes[:, 0] #Tenure
clientes[0,3] #Fila 1, columna 1
#clientes[0:2] #Filas 0 y 1 -> Los primero 2 clientes completos


np.float64(1.0)

Normalización, BroadsCasting, Axis y Min - Max


In [59]:
#BroadCasting = Aplicar una operacion entre un array grande y uno pqueño sin uso de bucles.
cargo_todos = clientes[:, 1]
cargo_cents = cargo_todos * 100 # Multiplicar cada elemento por 100
print(cargo_todos)
print(cargo_cents)

[29.85 56.85 45.85 60.85 70.85]
[2985. 5685. 4585. 6085. 7085.]


In [60]:
#Sin broadcasting (Lento y feo)
cargo_cents = []
for valor in cargo_todos:
    cargo_cents.append(valor * 100)

print(cargo_cents)

[np.float64(2985.0), np.float64(5685.0), np.float64(4585.0), np.float64(6085.0), np.float64(7084.999999999999)]


Axis = 0 y Axis =1


In [63]:
#Axis=0: La direccion de las filas de arriba hacia abajo
#Axis=1: la dirección de las columnas de izquierda a derecha
promedios_col = np.mean(clientes, axis=0)
print(promedios_col)

promedios_fil = np.mean(clientes, axis=1)
print(promedios_fil)

[ 17.    52.85 756.65   3.2    0.2 ]
[ 12.34 318.94  30.34 422.54  45.74]


In [66]:
#Normalización
#paso importante para entrenar un modelo de ML
#¿el problema? las features tienen escalas muy distintas
#Tenure = 0 y 72
#totalCharges: valores entre 0 y 8684
#Solucione: Llevar todo entre [0,1]

x_min = clientes.min(axis=0)
x_max = clientes.max(axis=0)
x_norm = (clientes - x_min) / (x_max - x_min) 
print(x_min)
print(x_max)
print()
print(x_norm)

[ 1.   29.85 29.85  1.    0.  ]
[  45.     70.85 2000.85    5.      1.  ]

[[0.   0.   0.   0.   0.  ]
 [0.75 0.66 0.75 0.5  0.  ]
 [0.02 0.39 0.04 0.5  0.  ]
 [1.   0.76 1.   1.   1.  ]
 [0.05 1.   0.06 0.75 0.  ]]
